In [2]:
import pdb
import time
import duckdb
import sqlite3
import logging
import datetime as dt
import tabulate as tab

## Funções Auxiliares

In [3]:
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

In [4]:
def dict_factory(cursor, row):
    fields = [column[0] for column in cursor.description]
    return {key: value for key, value in zip(fields, row)}

In [5]:
def get_time_id(cursor, date_str):
    if not date_str:
        return None

    row = cursor.execute(
        "select id from dim_time where dt = ?", (date_str[:13],)
    ).fetchone()

    return row["id"] if row else None

## Comandos DDL

In [6]:
def create_dim_time(conn: sqlite3.Connection):
    sql = """
    create table if not exists dim_time (
        id integer primary key, -- automatically auto increments
        dt datetime unique,
        year int,
        month int,
        day int,
        hour int,
        quarter int,
        day_name text,
        day_name_abbr text,
        month_name text,
        month_name_abbr text,
        is_weekend int,
        day_period text
    )
    """
    cur = conn.cursor()
    cur.execute(sql)

In [7]:
def create_dim_product(conn: sqlite3.Connection):
    sql = """
    -- Representa: Products, Categories e Suppliers
    -- Três opções de gerenciamento dos valores:
    -- primeira inserção é a que fica
    -- última inserção é a que fica (update)
    -- guardamos todos os estados (surrogate obrigatória)
    -- (product_id, product_unit_price) => histórico de variação
    -- dos preços do produto
    create table if not exists dim_product (
        id integer primary key, -- automatically auto increments
        product_id int unique,  -- chave natural
        product_name text,
        category_name text,
        category_description text,
        supplier_company text,
        supplier_city text,
        supplier_region text,
        supplier_country text,
        product_unit_price numeric,
        product_quantity_per_unit numeric,
        product_discontinued boolean
    )
    """
    cur = conn.cursor()
    cur.execute(sql)

In [8]:
def create_dim_customer(conn: sqlite3.Connection):
    sql = """
    create table if not exists dim_customer (
        id integer primary key, -- automatically auto increments
        customer_id text unique,       -- chave natural
        company_name text,
        contact_name text,
        city text,
        region text,
        country text
    )
    """
    cur = conn.cursor()
    cur.execute(sql)

In [9]:
def create_dim_employee(conn: sqlite3.Connection):
    sql = """
    create table if not exists dim_employee (
        id integer primary key, -- automatically auto increments
        employee_id int unique, -- chave natural
        full_name text,
        title text,
        birth_date date,
        hire_date date,
        city text,
        region text,
        country text,
        reports_to_name text
    )
    """
    cur = conn.cursor()
    cur.execute(sql)

In [10]:
def create_dim_shipper(conn: sqlite3.Connection):
    sql = """
    create table if not exists dim_shipper (
        id integer primary key, -- automatically auto increments
        shipper_id int unique,  -- chave natural
        company_name text,
        phone text
    )
    """
    cur = conn.cursor()
    cur.execute(sql)

In [11]:
def create_dim_ship_location(conn: sqlite3.Connection):
    sql = """
    create table if not exists dim_ship_location (
        id integer primary key, -- automatically auto increments
        city text,
        region text,
        country text,
        unique (city, region, country)
    )
    """
    cur = conn.cursor()
    cur.execute(sql)

In [12]:
def create_fact_sales(conn: sqlite3.Connection):
    sql = """
    -- Representa: Orders e Order Details
    create table if not exists fact_sales (
        id integer primary key, -- automatically auto increments
        order_id int,    -- chave natural
        -- Chaves Estrangeiras
        order_time_id int references dim_time(id),
        required_time_id int references dim_time(id),
        shipped_time_id int references dim_time(id) null,
        customer_id int references dim_customer(id),
        product_id int references dim_product(id),  -- chave natural
        employee_id int references dim_employee(id),
        shipper_id int references dim_shipper(id),
        ship_location_id int references dim_ship_location(id),
        -- Metricas
        unit_price_sold numeric,
        quantity int,
        discount numeric,
        revenue_net numeric, -- (Price * Qty) * (1 - Discount)
        freight_apportioned numeric, -- granularidade do campo diferente da granularidade da dimensão
        unique(order_id, product_id)
    )
    """
    cur = conn.cursor()
    cur.execute(sql)

## Lógica de Ingestão

In [13]:
def insert_dim_time(
    src_conn: sqlite3.Connection,
    dst_conn: sqlite3.Connection,
    dt_range: tuple[dt.datetime, dt.datetime] = None
):
    def fetch_start_end_dates():
        sql_dates = """
        select min(datetime(OrderDate)) as dt_start,
               max(datetime(OrderDate)) as dt_end
        from Orders
        """
        cur = src_conn.cursor()
        res = cur.execute(sql_dates).fetchone()
        return res["dt_start"], res["dt_end"]

    def calculate_day_period(hour: int):
        if 6 <= hour and hour <= 11:
            return "Morning"
        elif 12 <= hour and hour <= 17:
            return "Afternoon"
        elif 18 <= hour and hour <= 23:
            return "Night"
        else:
            return "Midnight"

    if not dt_range:
        dt_start, dt_end = fetch_start_end_dates()
        dt_start = dt.datetime.strptime(dt_start, "%Y-%m-%d %H:%M:%S")
        dt_end = dt.datetime.strptime(dt_end, "%Y-%m-%d %H:%M:%S")
    else:
        dt_start, dt_end = dt_range

    dt_aux = dt_start.replace(hour=0, minute=0, second=0)
    values = []
    i = 1
    while dt_aux <= dt_end:
        values.append(
            {
                "dt": dt_aux.strftime('%Y-%m-%d %H'),
                "year": dt_aux.year,
                "month": dt_aux.month,
                "day": dt_aux.day,
                "hour": dt_aux.hour,
                "quarter": (dt_aux.month - 1) // 3 + 1,
                "day_name": dt_aux.strftime("%A"),
                "day_name_abbr": dt_aux.strftime("%a"),
                "month_name": dt_aux.strftime("%B"),
                "month_name_abbr": dt_aux.strftime("%b"),
                "is_weekend": dt_aux.weekday() > 4,
                "day_period": calculate_day_period(dt_aux.hour),
            }
        )
        if len(values) % 10_000 == 0:
            logger.info(f"Inserted {i * len(values)} date times")
            i += 1
            cur = dst_conn.cursor()
            cur.executemany("""
                insert into dim_time (
                    dt, year, month, day, hour, quarter, day_name, day_name_abbr, month_name, month_name_abbr, is_weekend, day_period
                ) values (
                    :dt, :year, :month, :day, :hour, :quarter, :day_name, :day_name_abbr, :month_name, :month_name_abbr, :is_weekend, :day_period
                )
                """,
                values
            )
            dst_conn.commit()
            cur.close()
            values = []
        dt_aux += dt.timedelta(hours=1)
    if values:
        logger.info(f"Inserted {i * len(values) + len(values)} date times")
        cur = dst_conn.cursor()
        cur.executemany("""
            insert into dim_time (
                dt, year, month, day, hour, quarter, day_name, day_name_abbr, month_name, month_name_abbr, is_weekend, day_period
            ) values (
                :dt, :year, :month, :day, :hour, :quarter, :day_name, :day_name_abbr, :month_name, :month_name_abbr, :is_weekend, :day_period
            )
            """,
            values
        )
        dst_conn.commit()
        cur.close()
        values = []

In [14]:
def insert_dim_product(
    src_conn: sqlite3.Connection,
    dst_conn: sqlite3.Connection,
):
    def fetch_joined_info():
        sql = """
        select
            p.ProductID as product_id,
            p.ProductName as product_name,
            c.CategoryName as category_name,
            c.Description as category_description,
            s.CompanyName as supplier_company,
            s.City as supplier_city,
            s.Region as supplier_region,
            s.Country as supplier_country,
            p.UnitPrice as product_unit_price,
            p.QuantityPerUnit as product_quantity_per_unit,
            case
                when p.Discontinued = '0'
                    then false
                when p.Discontinued = '1'
                    then true
                else false
            end as product_discontinued
        from Products p
        inner join Categories c on p.CategoryID = c.CategoryID
        inner join Suppliers s on p.SupplierID = s.SupplierID
        """
        cur = src_conn.cursor()
        res = cur.execute(sql).fetchall()
        return res

    values = fetch_joined_info()
    cur = dst_conn.cursor()
    cur.executemany("""
        insert into dim_product (
            product_id, product_name, category_name, category_description, supplier_company, supplier_city, supplier_region, supplier_country, product_unit_price,
            product_quantity_per_unit, product_discontinued
        ) values (
            :product_id, :product_name, :category_name, :category_description, :supplier_company, :supplier_city, :supplier_region, :supplier_country, :product_unit_price,
            :product_quantity_per_unit, :product_discontinued
        )
        """,
        values
    )
    dst_conn.commit()
    cur.close()

In [15]:
def insert_dim_customer(
    src_conn: sqlite3.Connection,
    dst_conn: sqlite3.Connection,
):
    def fetch_joined_info():
        sql = """
        select
            CustomerID as customer_id,
            CompanyName as company_name,
            ContactName as contact_name,
            City as city,
            Region as region,
            Country as country
        from Customers
        """
        cur = src_conn.cursor()
        res = cur.execute(sql).fetchall()
        return res

    values = fetch_joined_info()
    cur = dst_conn.cursor()
    cur.executemany("""
        insert into dim_customer (
            customer_id, company_name, contact_name, city, region, country
        ) values (
            :customer_id, :company_name, :contact_name, :city, :region, :country
        )
        """,
        values
    )
    dst_conn.commit()
    cur.close()

In [16]:
def insert_dim_employee(
    src_conn: sqlite3.Connection,
    dst_conn: sqlite3.Connection,
):
    def fetch_joined_info():
        sql = """
        select
            e1.EmployeeID as employee_id,
            e1.FirstName || ' ' || e1.LastName as full_name,
            e1.Title as title,
            datetime(e1.BirthDate) as birth_date,
            datetime(e1.HireDate) as hire_date,
            e1.City as city,
            e1.Region as region,
            e1.Country as country,
            e2.FirstName || ' ' || e2.LastName as reports_to_name
        from Employees e1
            left outer join Employees e2 on e1.ReportsTo = e2.EmployeeID
        """
        cur = src_conn.cursor()
        res = cur.execute(sql).fetchall()
        return res

    values = fetch_joined_info()
    cur = dst_conn.cursor()
    cur.executemany("""
        insert into dim_employee (
            employee_id, full_name, title, birth_date, hire_date, city,
            region, country, reports_to_name
        ) values (
            :employee_id, :full_name, :title, :birth_date, :hire_date, :city,
            :region, :country, :reports_to_name
        )
        """,
        values
    )
    dst_conn.commit()
    cur.close()

In [17]:
def insert_dim_shipper(
    src_conn: sqlite3.Connection,
    dst_conn: sqlite3.Connection,
):
    def fetch_joined_info():
        sql = """
        select
            ShipperID as shipper_id,
            CompanyName as company_name
        from Shippers
        """
        cur = src_conn.cursor()
        res = cur.execute(sql).fetchall()
        return res

    values = fetch_joined_info()
    cur = dst_conn.cursor()
    cur.executemany("""
        insert into dim_shipper (
            shipper_id, company_name
        ) values (
            :shipper_id, :company_name
        )
        """,
        values
    )
    dst_conn.commit()
    cur.close()

In [18]:
def insert_dim_ship_location(
    src_conn: sqlite3.Connection,
    dst_conn: sqlite3.Connection,
):
    def fetch_joined_info():
        sql = """
        select distinct
            ShipCity as city,
            ShipRegion as region,
            ShipCountry as country
        from Orders
        """
        cur = src_conn.cursor()
        res = cur.execute(sql).fetchall()
        return res

    values = fetch_joined_info()
    cur = dst_conn.cursor()
    cur.executemany("""
        insert into dim_ship_location (
            city, region, country
        ) values (
            :city, :region, :country
        )
        """,
        values
    )
    dst_conn.commit()
    cur.close()

In [19]:
def insert_fact_sales(
    src_conn: sqlite3.Connection,
    dst_conn: sqlite3.Connection,
):
    def fetch_joined_info():
        sql = """
        select
            o.OrderID as order_id,
            datetime(o.OrderDate) as order_dt,
            datetime(o.RequiredDate) as required_dt,
            datetime(o.ShippedDate) as shipped_dt,
            o.CustomerID as customer_id,
            od.ProductID as product_id,
            o.EmployeeID as employee_id,
            o.ShipVia as shipper_id,
            o.ShipCity as ship_city,
            o.ShipRegion as ship_region,
            o.ShipCountry as ship_country,
            od.UnitPrice as unity_price_sold,
            od.Quantity as quantity,
            od.Discount as discount,
            round((od.UnitPrice * od.Quantity) * (1 - od.Discount), 2) as revenue_net,
            0 as freight_apportioned
        from Orders o
        inner join main."Order Details" od on o.OrderID = od.OrderID
        """
        cur = src_conn.cursor()
        res = cur.execute(sql).fetchall()
        return res

    values = fetch_joined_info()
    dst_cur = dst_conn.cursor()
    final_values = []
    i = 0
    for value in values:
        if len(final_values) > 0 and len(final_values) % 10_000 == 0:
            i += 1
            logger.info(f"{i}: Inserted fact values")
            cur = dst_conn.cursor()
            cur.executemany("""
                insert into fact_sales (
                    order_id,
                    order_time_id,
                    required_time_id,
                    shipped_time_id,
                    customer_id,
                    product_id,
                    employee_id,
                    shipper_id,
                    ship_location_id,
                    unit_price_sold,
                    quantity,
                    discount,
                    revenue_net,
                    freight_apportioned
                ) values (
                    ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?
                )
                """,
                final_values
            )
            dst_conn.commit()
            cur.close()
            final_values = []
        # Operações Lookup
        order_time_id = get_time_id(dst_cur, value["order_dt"])
        required_time_id = get_time_id(dst_cur, value["required_dt"])
        shipped_time_id = get_time_id(dst_cur, value["shipped_dt"])
        customer_id = dst_cur.execute(
            "select id from dim_customer where customer_id = ?", (value["customer_id"],)
        ).fetchone()
        product_id = dst_cur.execute(
            "select id from dim_product where product_id = ?", (value["product_id"],)
        ).fetchone()
        employee_id = dst_cur.execute(
            "select id from dim_employee where employee_id = ?", (value["employee_id"],)
        ).fetchone()
        shipper_id = dst_cur.execute(
            "select id from dim_shipper where shipper_id = ?", (value["shipper_id"],)
        ).fetchone()
        ship_location_id = dst_cur.execute(
            "select id from dim_ship_location where city = ? and region = ? and country = ?", (value["ship_city"], value["ship_region"], value["ship_country"])
        ).fetchone()
        final_values.append(
            (
                value["order_id"],
                order_time_id,
                required_time_id,
                shipped_time_id,
                customer_id["id"],
                product_id["id"],
                employee_id["id"],
                shipper_id["id"],
                ship_location_id["id"],
                value["unity_price_sold"],
                value["quantity"],
                value["discount"],
                value["revenue_net"],
                0
            )
        )
    if final_values:
        logger.info(f"{i}: Inserted fact values")
        cur = dst_conn.cursor()
        cur.executemany("""
            insert into fact_sales (
                order_id,
                order_time_id,
                required_time_id,
                shipped_time_id,
                customer_id,
                product_id,
                employee_id,
                shipper_id,
                ship_location_id,
                unit_price_sold,
                quantity,
                discount,
                revenue_net,
                freight_apportioned
            ) values (
                ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?
            )
            """,
            final_values
        )
        dst_conn.commit()
        cur.close()
        final_values = []
    dst_conn.commit()
    dst_cur.close()

In [20]:
with sqlite3.connect("northwind.db") as oltp_conn, sqlite3.connect("northwind-dw.db") as olap_conn:
    oltp_conn.row_factory = dict_factory
    olap_conn.row_factory = dict_factory
    # DDL
    create_dim_time(olap_conn)
    create_dim_product(olap_conn)
    create_dim_customer(olap_conn)
    create_dim_employee(olap_conn)
    create_dim_shipper(olap_conn)
    create_dim_ship_location(olap_conn)
    create_fact_sales(olap_conn)
    # ETL
    # insert_dim_time(oltp_conn, olap_conn)
    # insert_dim_product(oltp_conn, olap_conn)
    # insert_dim_customer(oltp_conn, olap_conn)
    # insert_dim_employee(oltp_conn, olap_conn)
    # insert_dim_shipper(oltp_conn, olap_conn)
    # insert_dim_ship_location(oltp_conn, olap_conn)
    # insert_fact_sales(oltp_conn, olap_conn)

## Relatórios OLTP vs OLAP

In [49]:
def run_query(conn: sqlite3.Connection | duckdb.DuckDBPyConnection, query: str):
    if isinstance(conn, sqlite3.Connection):
        conn.row_factory = dict_factory

    cur = conn.cursor()
    exec = cur.execute(query)

    if isinstance(conn, duckdb.DuckDBPyConnection):
        rows = exec.fetchall()
        cols = [col[0] for col in exec.description]
        result = [dict(zip(cols, row)) for row in rows]
    else:
        result = exec.fetchall()

    cur.close()
    return result

In [22]:
def cronometrar(func, *args):
    start_time = time.perf_counter()
    ret = func(*args)
    end_time = time.perf_counter()
    duration = end_time - start_time
    print(f"Executed in {duration:.4f} seconds")
    return ret

In [56]:
def tabular(
    data: list[dict],
    n: int = 10,
    headers: list[str] | None = None,
    tablefmt: str = "simple",
):
    display_headers = headers if headers else "keys"
    print(
        tab.tabulate(
            data if n == -1 else data[:n],
            headers=display_headers,
            tablefmt=tablefmt
        )
    )

### Category Sales for 2013

In [24]:
oltp_query = """
SELECT Categories.CategoryName,
       Products.ProductName,
       round(Sum(([Order Details].UnitPrice*Quantity*(1-Discount)/100)*100), 2) AS ProductSales
FROM Categories
 JOIN    Products On Categories.CategoryID = Products.CategoryID
    JOIN  [Order Details] on Products.ProductID = [Order Details].ProductID
     JOIN  [Orders] on Orders.OrderID = [Order Details].OrderID
WHERE date(Orders.ShippedDate) Between date('2013-01-01') And date('2013-12-31')
GROUP BY Categories.CategoryName, Products.ProductName
"""

In [25]:
olap_query = """
select dp.category_name,
       dp.product_name,
       round(sum(fs.revenue_net), 2) as product_sales
from fact_sales fs
inner join dim_product dp on dp.id = fs.product_id
inner join dim_time dt on dt.id = fs.shipped_time_id
where dt.year = 2013
group by dp.category_name, dp.product_name
"""

In [26]:
with sqlite3.connect("northwind.db") as oltp_conn, sqlite3.connect("northwind-dw.db") as olap_conn:
    oltp_conn.row_factory = dict_factory
    olap_conn.row_factory = dict_factory
    tabular(cronometrar(run_query, oltp_conn, oltp_query))
    print("=" * 50)
    tabular(cronometrar(run_query, olap_conn, olap_query))

Executed in 0.2167 seconds
CategoryName    ProductName                    ProductSales
--------------  -------------------------  ----------------
Beverages       Chai                       321552
Beverages       Chang                      334628
Beverages       Chartreuse verte           310680
Beverages       Côte de Blaye                   4.42548e+06
Beverages       Guaraná Fantástica          77638.5
Beverages       Ipoh Coffee                774548
Beverages       Lakkalikööri               290124
Beverages       Laughing Lumberjack Lager  248948
Beverages       Outback Lager              264930
Beverages       Rhönbräu Klosterbier       141716
Executed in 0.4193 seconds
category_name    product_name                  product_sales
---------------  -------------------------  ----------------
Beverages        Chai                       321552
Beverages        Chang                      334628
Beverages        Chartreuse verte           310680
Beverages        Côte de Blaye         

### Amount Orders per Quarter

In [27]:
oltp_query = """
SELECT strftime('%Y', Orders.OrderDate) as year,
       CASE WHEN cast(strftime('%m', Orders.OrderDate) as int) between 1 and 3 then 1
            WHEN cast(strftime('%m', Orders.OrderDate) as int) between 4 and 6 then 2
            WHEN cast(strftime('%m', Orders.OrderDate) as int) between 7 and 9 then 3
            WHEN cast(strftime('%m', Orders.OrderDate) as int) between 10 and 12 then 4
            ELSE 0
       END as quarter,
       count(Orders.OrderID) as amount_orders
FROM [Orders]
  JOIN  [Order Details] on Orders.OrderID = [Order Details].OrderID
GROUP BY 1, 2
"""

In [28]:
olap_query = """
select dt.year,
       dt.quarter,
       count(fs.order_id) as amount_orders
from fact_sales fs
inner join dim_time dt on fs.order_time_id = dt.id
group by dt.year, dt.quarter
"""

In [29]:
with sqlite3.connect("northwind.db") as oltp_conn, sqlite3.connect("northwind-dw.db") as olap_conn:
    oltp_conn.row_factory = dict_factory
    olap_conn.row_factory = dict_factory
    tabular(cronometrar(run_query, oltp_conn, oltp_query))
    print("=" * 50)
    tabular(cronometrar(run_query, olap_conn, olap_query))

Executed in 0.5988 seconds
  year    quarter    amount_orders
------  ---------  ---------------
  2012          3            12305
  2012          4            13216
  2013          1            12998
  2013          2            13289
  2013          3            13721
  2013          4            12823
  2014          1            12518
  2014          2            14068
  2014          3            12596
  2014          4            13818
Executed in 0.5141 seconds
  year    quarter    amount_orders
------  ---------  ---------------
  2012          3            12305
  2012          4            13216
  2013          1            12998
  2013          2            13289
  2013          3            13721
  2013          4            12823
  2014          1            12518
  2014          2            14068
  2014          3            12596
  2014          4            13818


## Introdução ao DuckDB

### Porque usarmos o DuckDB?

**"A ferramenta certa para a tarefa de análise"**

Mesmo com os dados estando persistidos no arquivo `SQLite`, o `DuckDB` age como um motor que processa consultas analíticas (expansão das funcionalidades do `SQLite`).

---

### 1. Arquitetura semelhante de ambos

Ambos usam um único arquivo para manusear, mas tem objetivos distintos::

*   **SQLite:** É otimizado para o universo transacional – **OLTP** (Online Transactional Processing). Os dados são armazenados em in **linhas**.
*   **DuckDB:** É otimizado para o universo analítico – **OLAP** (Online Analytical Processing). Os dados são armazenados e processados em **colunas**.

---

### 2. Processamento Linha vs. Colunas

*   **SQLite (baseado em linhas):** Para calcular o total de vendas em 2023, o processamento lê a linha inteira (`id`, `date`, `product`, `quantity`, `price`, `discount`, ...) para utilizar apenas a coluna `price`. Isso consome memória e I/O.
*   **DuckDB (baseado em colunas):** Ele processa diretamente as colunas `price` e `date`, ignorando as demais. Esse é o motivo do `DuckDB` ser muito mais rápido em consultas analíticas.

---

### 3. Processamento analítico local

Reforçando que o `DuckDB` não está substituindo o `SQLite`, o está estendendo.

Ao usar o comando `duckdb.connect("sqlite:northwind-dw.db")`, estamos usando o `DuckDB` como um **motor de query em cima do armazenamento do SQLite**.
*   Ganhamos a **portabilidade** do `SQLite` (um arquivo).
*   Ganhamos o **poder de processamento** do `DuckDB` (SQL avançado com comandos `ROLLUP`, `GROUPING SETS` e `Window Functions`).

---

### Workflow construído

1.  **SQLite (Fonte):** Onde os dados OLTP residem.
2.  **Python (ETL):** A ponte que limpa, normaliza e redesenha os dados no **Modelo Estrela**.
3.  **DuckDB (Analítico):** O motor (em cima do `SQLite`) que transforma a Modelagem Estrela em `insights` and `relatórios`.

### Quantidade de Pedidos por períodos

- The Grand Total: (None, None, None) — A soma total.
- Yearly Subtotals: (2023, None, None) — Total para o ano de 2023.
- Quarterly Subtotals: (2023, 4, None) — Total para o Q4 de 2023.
- Monthly Granularity: (2023, 4, 12) — Nível Grão ou Folha.

In [61]:
duck_olap_query = """
select dt.year, dt.quarter, dt.month, count(fs.order_id) as total_orders
from fact_sales fs
inner join dim_time dt ON fs.order_time_id = dt.id
group by rollup (dt.year, dt.quarter, dt.month)
order by 1, 2, 3
"""

In [62]:
with duckdb.connect("sqlite:northwind-dw.db") as olap_conn:
    tabular(cronometrar(run_query, olap_conn, duck_olap_query), n=-1)

Executed in 0.3806 seconds
  year    quarter    month    total_orders
------  ---------  -------  --------------
  2012          3        7            2821
  2012          3        8            4773
  2012          3        9            4711
  2012          3                    12305
  2012          4       10            4410
  2012          4       11            3945
  2012          4       12            4861
  2012          4                    13216
  2012                               25521
  2013          1        1            4238
  2013          1        2            4033
  2013          1        3            4727
  2013          1                    12998
  2013          2        4            4413
  2013          2        5            4814
  2013          2        6            4062
  2013          2                    13289
  2013          3        7            5863
  2013          3        8            4124
  2013          3        9            3734
  2013          3          

### Quantidade de Pedidos por localidade

- The Grand Total: (None, None, None) — A soma total.
- Region Subtotals: (South America, None, None) — Total para América do Sul.
- Country Subtotals: (South America, Brazil, None) — Total para o Brasil, integrante da América do Sul.
- City Granularity: (South America, Brazil, Rio de Janeiro) — Nível Grão ou Folha.

In [71]:
duck_olap_query = """
select dc.region, dc.country, dc.city, count(fs.order_id) as total_orders
from fact_sales fs
inner join dim_customer dc ON fs.customer_id = dc.id
group by rollup (dc.region, dc.country, dc.city)
order by 1, 2, 3
"""

In [72]:
with duckdb.connect("sqlite:northwind-dw.db") as olap_conn:
    tabular(cronometrar(run_query, olap_conn, duck_olap_query), n=-1)

Executed in 0.3428 seconds
region           country      city               total_orders
---------------  -----------  ---------------  --------------
British Isles    Ireland      Cork                       6442
British Isles    Ireland                                 6442
British Isles    UK           Cowes                      6598
British Isles    UK           London                    39457
British Isles    UK                                     46055
British Isles                                           52497
Central America  Mexico       México D.F.               33783
Central America  Mexico                                 33783
Central America                                         33783
Eastern Europe   Poland       Warszawa                   6448
Eastern Europe   Poland                                  6448
Eastern Europe                                           6448
North America    Canada       Montréal                   6681
North America    Canada       Tsawassen    

### Category Sales for 2013 (destaque tempo de execução)

In [65]:
duck_olap_query = """
select dp.category_name,
       dp.product_name,
       round(sum(fs.revenue_net), 2) as product_sales
from fact_sales fs
inner join dim_product dp on dp.id = fs.product_id
inner join dim_time dt on dt.id = fs.shipped_time_id
where dt.year = 2013
group by dp.category_name, dp.product_name
"""

In [70]:
with duckdb.connect("sqlite:northwind-dw.db") as olap_conn:
    tabular(cronometrar(run_query, olap_conn, duck_olap_query))

Executed in 0.3685 seconds
  year    quarter    amount_orders
------  ---------  ---------------
  2021          1            13294
  2019          3            12501
  2022          2            13332
  2013          3            13721
  2020          3            15226
  2023          2            13946
  2021          4            14713
  2017          1            13276
  2018          3            12937
  2013          2            13289


### Amount Orders per Quarter (destaque tempo de execução)

In [67]:
duck_olap_query = """
select dt.year,
       dt.quarter,
       count(fs.order_id) as amount_orders
from fact_sales fs
inner join dim_time dt on fs.order_time_id = dt.id
group by dt.year, dt.quarter
"""

In [69]:
with duckdb.connect("sqlite:northwind-dw.db") as olap_conn:
    tabular(cronometrar(run_query, olap_conn, duck_olap_query))

Executed in 0.3842 seconds
  year    quarter    amount_orders
------  ---------  ---------------
  2021          3            14450
  2013          4            12823
  2020          4            12219
  2019          1            12429
  2021          1            13294
  2019          3            12501
  2022          2            13332
  2013          3            13721
  2020          3            15226
  2023          2            13946
